# Spark DataFrames & Spark SQL


In [ ]:
# Scenario Statement: Global Streaming Platform OptimizationThe Context: You are a Data Engineer at a global video
# streaming service (like Netflix or YouTube). Every second, your servers generate logs for millions of active video streams.
# These logs contain the "Bandwidth Allocation" (how much data is being sent) for every single user currently watching a video.

# The Problem: Due to a new "Ultra-HD" initiative, the company needs to double the bandwidth allocation for every active stream
#  immediately to improve video quality.

# The Challenge:Massive Data: The number of logs is in the millions, making it impossible for
#   a single computer to process them without crashing or taking hours.

# # Real-Time Speed: The infrastructure costs need to be calculated instantly so server capacity can be adjusted in real-time.
# Mapping the Scenario to the CodeCode StepScenario



from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LabSolution").getOrCreate()

Load Dataset

In [4]:
from google.colab import files
uploaded = files.upload()



Saving spark_lab_dataset (1).csv to spark_lab_dataset (1) (1).csv
   transaction_id  customer_id              name region    amount  \
0               1         1057  Christian Forbes  South  12935.47   
1               2         1003     Justin Harris  South   3709.30   
2               3         1004     Peter Schmitt  South   3837.45   
3               4         1049    Molly Gonzalez   East  46638.10   
4               5         1045   Wanda Mcfarland  North  24212.31   

  transaction_type  is_fraud  
0         Purchase         0  
1       Withdrawal         0  
2         Purchase         1  
3       Withdrawal         1  
4         Purchase         1  


In [5]:
import pandas as pd
df = pd.read_csv("spark_lab_dataset (1).csv")
print(df.head())

   transaction_id  customer_id              name region    amount  \
0               1         1057  Christian Forbes  South  12935.47   
1               2         1003     Justin Harris  South   3709.30   
2               3         1004     Peter Schmitt  South   3837.45   
3               4         1049    Molly Gonzalez   East  46638.10   
4               5         1045   Wanda Mcfarland  North  24212.31   

  transaction_type  is_fraud  
0         Purchase         0  
1       Withdrawal         0  
2         Purchase         1  
3       Withdrawal         1  
4         Purchase         1  


Data Exploration

In [13]:
print("Total Transactions:", df.count())
df.select("region").distinct().show()
df.groupBy("transaction_type").count().show()

Total Transactions: 500
+------+
|region|
+------+
| South|
|  East|
|  West|
| North|
+------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



In [7]:
df = spark.createDataFrame(df)

print("Total Transactions:", df.count())
df.select("region").distinct().show()
df.groupBy("transaction_type").count().show()

Total Transactions: 500
+------+
|region|
+------+
| South|
|  East|
|  West|
| North|
+------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



Data Cleaning

In [8]:
df = df.dropna()
df.printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- is_fraud: long (nullable = true)



Transformations

In [9]:
high_value = df.filter(df.amount > 10000)
fraud_df = df.filter(df.is_fraud == 1)
selected = df.select("customer_id", "amount", "region")

high_value.show(5)
fraud_df.show(5)
selected.show(5)

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
|             6|       1045|  Jennifer Casey| South|35929.02|      Withdrawal|       1|
|             7|       1054|   Melissa Klein| North|26482.51|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
+--------------+-----------+---------------+------+--------+----------------+--------+
|transaction_id|customer_id|           name|region|  amount|transaction_type|is_fraud|
+---------

Aggregations

In [10]:
df.groupBy("region").avg("amount").show()
df.groupBy("region").sum("is_fraud").show()
df.groupBy("transaction_type").count().show()

+------+------------------+
|region|       avg(amount)|
+------+------------------+
| South| 24863.82100775194|
|  East|23983.666198347106|
|  West|25563.426111111115|
| North|23597.939032258062|
+------+------------------+

+------+-------------+
|region|sum(is_fraud)|
+------+-------------+
| South|           57|
|  East|           59|
|  West|           58|
| North|           67|
+------+-------------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



Spark SQL

In [11]:
df.createOrReplaceTempView("transactions")

spark.sql("SELECT region, COUNT(*) as total FROM transactions GROUP BY region").show()

spark.sql("SELECT * FROM transactions ORDER BY amount DESC LIMIT 5").show()

spark.sql("SELECT region, SUM(is_fraud) as fraud_count FROM transactions GROUP BY region").show()

+------+-----+
|region|total|
+------+-----+
| South|  129|
|  East|  121|
|  West|  126|
| North|  124|
+------+-----+

+--------------+-----------+-----------------+------+--------+----------------+--------+
|transaction_id|customer_id|             name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+-----------------+------+--------+----------------+--------+
|           427|       1036|      Todd Thomas|  West|49969.04|      Withdrawal|       1|
|           235|       1025|       Mark Smith| South|49825.23|          Refund|       0|
|           311|       1081|          Alan Yu|  West|49818.54|        Purchase|       0|
|           117|       1056|     Thomas Smith| South|49629.78|          Refund|       1|
|           305|       1046|Elizabeth Krueger| North|49602.85|        Purchase|       0|
+--------------+-----------+-----------------+------+--------+----------------+--------+

+------+-----------+
|region|fraud_count|
+------+-----------+
| South|      

Optimization

In [12]:
df.cache()
df.explain()

== Physical Plan ==
InMemoryTableScan [transaction_id#0L, customer_id#1L, name#2, region#3, amount#4, transaction_type#5, is_fraud#6L]
   +- InMemoryRelation [transaction_id#0L, customer_id#1L, name#2, region#3, amount#4, transaction_type#5, is_fraud#6L], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Filter atleastnnonnulls(7, transaction_id#0L, customer_id#1L, name#2, region#3, amount#4, transaction_type#5, is_fraud#6L)
            +- *(1) Scan ExistingRDD[transaction_id#0L,customer_id#1L,name#2,region#3,amount#4,transaction_type#5,is_fraud#6L]


